# DataMind Survey: Run Experiments 1-3 on Colab Pro

Use this notebook for the feasible project scope: Exp. 1, Exp. 2, and Exp. 3. It avoids OpenAI API quota by using a local deterministic judge and serves Qwen2.5-7B with vLLM on the Colab GPU.

Before running: in Colab, click `Runtime` -> `Change runtime type` -> choose `GPU`. Prefer `A100` or `L4`; `T4` can work for smaller `N_SAMPLES` but may be slow.

In [13]:
# Configuration
REPO_URL = "https://github.com/Razim021/datamind-survey.git"
BRANCH = "main"

N_SAMPLES = 50
MODEL_NAME = "Qwen2.5-7B-Instruct"
HF_MODEL = "Qwen/Qwen2.5-7B-Instruct"
API_PORT = 8000
MAX_ROUNDS = 6

print("Configured for", MODEL_NAME, "with", N_SAMPLES, "samples")

Configured for Qwen2.5-7B-Instruct with 5 samples


In [14]:
# Check GPU
!nvidia-smi

Sat Apr 25 15:45:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [15]:
# Clone project and install dependencies
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/datamind-survey")
if PROJECT_DIR.exists() and not (PROJECT_DIR / "requirements.txt").exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    result = subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr)
        raise RuntimeError(
            "Could not clone the repo. Most likely the GitHub repo is private. "
            "Make https://github.com/Razim021/datamind-survey public, then restart the Colab runtime."
        )

%cd /content/datamind-survey

# Upstream dependency used by the wrapper scripts.
if not Path("Datamind-main").exists():
    subprocess.check_call(["git", "clone", "https://github.com/zjunlp/DataMind", "Datamind-main"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "vllm"])
subprocess.check_call([sys.executable, "download_data.py"])

os.environ["DATAMIND_ROOT"] = "/content/datamind-survey/Datamind-main"
os.environ["DATA_DIR"] = "/content/datamind-survey/data"
os.environ["JUDGE_BACKEND"] = "local"
print("Setup complete")

/content/datamind-survey
Setup complete


In [16]:
# Sanity-check data loaders
!python - <<'PY'
from experiments.utils.data_loader import load_qrdata, load_discoverybench
print('QRData samples:', len(load_qrdata('data')))
print('DiscoveryBench samples:', len(load_discoverybench('data')))
PY

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
Loaded 411 QRData samples from data/QRData/benchmark/QRData.json
QRData samples: 411
Loaded 239 DiscoveryBench real-world samples from data/DiscoveryBench
DiscoveryBench samples: 239


NameError: name 'PY' is not defined

In [17]:
# Start vLLM server in the background
import subprocess
import time
import urllib.request
from pathlib import Path

log_path = Path('/content/vllm.log')
server_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', HF_MODEL,
    '--served-model-name', MODEL_NAME,
    '--port', str(API_PORT),
    '--dtype', 'float16',
    '--max-model-len', '4096',
]

server_log = open(log_path, 'w')
vllm_proc = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)
print('Started vLLM process:', vllm_proc.pid)
print('Log:', log_path)

for i in range(90):
    try:
        with urllib.request.urlopen(f'http://localhost:{API_PORT}/v1/models', timeout=2) as response:
            print('vLLM is ready')
            break
    except Exception:
        time.sleep(5)
else:
    print('vLLM did not become ready. Last log lines:')
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-40:]))

Started vLLM process: 8540
Log: /content/vllm.log
vLLM is ready


In [18]:
# Experiment 1: data-comprehension metadata ablation
%cd /content/datamind-survey/experiments
!python run_exp1_comprehension.py \
  --model_name {MODEL_NAME} \
  --data_dir ../data \
  --dataset qrdata \
  --sub_experiment info \
  --n_samples {N_SAMPLES} \
  --api_port {API_PORT} \
  --max_rounds {MAX_ROUNDS} \
  --judge_backend local \
  --output_dir results/exp1_colab

/content/datamind-survey/experiments
Loaded 411 QRData samples from ../data/QRData/benchmark/QRData.json

Dataset: QRDATA  |  Model: Qwen2.5-7B-Instruct

[Sub-exp A] Condition: w/o Info
Saved 5 results to results/exp1_colab/qrdata_wo_info_Qwen2.5-7B-Instruct.json
[qrdata | w/o Info]  Accuracy: 0.00%  |  Code Error Rate: 20.00%  |  N=5

[Sub-exp A] Condition: w/ Info
Saved 5 results to results/exp1_colab/qrdata_w_info_Qwen2.5-7B-Instruct.json
[qrdata | w/ Info]  Accuracy: 40.00%  |  Code Error Rate: 0.00%  |  N=5


=== EXPERIMENT 1 SUMMARY ===
  qrdata_without_metadata: Acc=0.0%  CodeErr=20.0%  N=5
  qrdata_with_metadata: Acc=40.0%  CodeErr=0.0%  N=5
  qrdata_info_delta: delta = +40.00 pp

Summary saved to results/exp1_colab/summary.json


In [19]:
# Experiment 2: code-generation and error analysis
!python run_exp2_code_analysis.py \
  --mode evaluate \
  --models {MODEL_NAME} \
  --data_dir ../data \
  --dataset qrdata \
  --n_samples {N_SAMPLES} \
  --api_port {API_PORT} \
  --max_rounds {MAX_ROUNDS} \
  --judge_backend local \
  --output_dir results/exp2_colab

Loaded 411 QRData samples from ../data/QRData/benchmark/QRData.json

[EVAL] Model=Qwen2.5-7B-Instruct  Dataset=QRDATA
Saved 5 results to results/exp2_colab/qrdata_Qwen2.5-7B-Instruct.json
[Qwen2.5-7B-Instruct | qrdata]  Accuracy: 0.00%  |  Code Error Rate: 20.00%  |  N=5


=== EXPERIMENT 2 SUMMARY ===
  Qwen2.5-7B-Instruct_qrdata: Acc=0.0%  CodeErr=20.0%


In [20]:
# Experiment 2b: local heuristic error categorization
!python run_exp2_code_analysis.py \
  --mode categorize \
  --results_dir results/exp2_colab \
  --n_errors {N_SAMPLES} \
  --judge_backend local \
  --output_dir results/exp2_colab


Categorizing 5 error samples ...

=== ERROR CATEGORIZATION RESULTS ===
  Planning & Reasoning Error: 0/5 (0.0%)
  Data Understanding Error: 4/5 (80.0%)
  Code Error: 1/5 (20.0%)

Saved to results/exp2_colab/error_categories.json


In [21]:
# Experiment 3: turn-budget study
!python run_exp3_turn_budget.py \
  --model_name {MODEL_NAME} \
  --data_dir ../data \
  --dataset qrdata \
  --turn_budgets 2,4,6 \
  --n_samples {N_SAMPLES} \
  --api_port {API_PORT} \
  --judge_backend local \
  --output_dir results/exp3_colab

Loaded 411 QRData samples from ../data/QRData/benchmark/QRData.json

[EXP3] Model=Qwen2.5-7B-Instruct Dataset=QRDATA Turn budget=2
Saved 5 results to results/exp3_colab/qrdata_turns2_Qwen2.5-7B-Instruct.json
[qrdata | turns=2]  Accuracy: 0.00%  |  Code Error Rate: 20.00%  |  N=5

[EXP3] Model=Qwen2.5-7B-Instruct Dataset=QRDATA Turn budget=4
Saved 5 results to results/exp3_colab/qrdata_turns4_Qwen2.5-7B-Instruct.json
[qrdata | turns=4]  Accuracy: 0.00%  |  Code Error Rate: 20.00%  |  N=5

[EXP3] Model=Qwen2.5-7B-Instruct Dataset=QRDATA Turn budget=6
Saved 5 results to results/exp3_colab/qrdata_turns6_Qwen2.5-7B-Instruct.json
[qrdata | turns=6]  Accuracy: 0.00%  |  Code Error Rate: 20.00%  |  N=5

=== EXPERIMENT 3 SUMMARY ===
  qrdata_turns_2: Acc=0.0% CodeErr=20.0%
  qrdata_turns_4: Acc=0.0% CodeErr=20.0%
  qrdata_turns_6: Acc=0.0% CodeErr=20.0%

Summary saved to results/exp3_colab/summary.json


In [22]:
# Show result summaries
!find results -maxdepth 3 -type f \( -name 'summary.json' -o -name 'error_categories.json' \) -print -exec python -m json.tool {} \;

results/exp3_colab/summary.json
{
    "qrdata_turns_2": {
        "accuracy": 0.0,
        "code_error_rate": 20.0,
        "n": 5
    },
    "qrdata_turns_4": {
        "accuracy": 0.0,
        "code_error_rate": 20.0,
        "n": 5
    },
    "qrdata_turns_6": {
        "accuracy": 0.0,
        "code_error_rate": 20.0,
        "n": 5
    }
}
results/exp2_colab/summary.json
{
    "Qwen2.5-7B-Instruct_qrdata": {
        "accuracy": 0.0,
        "code_error_rate": 20.0,
        "n": 5
    }
}
results/exp2_colab/error_categories.json
{
    "n_total": 5,
    "categories": {
        "planning_reasoning": {
            "count": 0,
            "percent": 0.0
        },
        "data_understanding": {
            "count": 4,
            "percent": 80.0
        },
        "code_error": {
            "count": 1,
            "percent": 20.0
        }
    }
}
results/exp1_colab/summary.json
{
    "qrdata_without_metadata": {
        "accuracy": 0.0,
        "code_error_rate": 20.0,
        "n": 

In [23]:
# Zip results for download
%cd /content/datamind-survey/experiments
!zip -r /content/datamind_first3_results.zip results
print('Download /content/datamind_first3_results.zip from the Colab file browser.')

/content/datamind-survey/experiments
  adding: results/ (stored 0%)
  adding: results/exp3_colab/ (stored 0%)
  adding: results/exp3_colab/qrdata_turns6_Qwen2.5-7B-Instruct.json (deflated 84%)
  adding: results/exp3_colab/qrdata_turns4_Qwen2.5-7B-Instruct.json (deflated 84%)
  adding: results/exp3_colab/summary.json (deflated 69%)
  adding: results/exp3_colab/qrdata_turns2_Qwen2.5-7B-Instruct.json (deflated 84%)
  adding: results/exp2_colab/ (stored 0%)
  adding: results/exp2_colab/qrdata_Qwen2.5-7B-Instruct.json (deflated 84%)
  adding: results/exp2_colab/summary.json (deflated 17%)
  adding: results/exp2_colab/error_categories.json (deflated 53%)
  adding: results/exp1_colab/ (stored 0%)
  adding: results/exp1_colab/qrdata_wo_info_Qwen2.5-7B-Instruct.json (deflated 84%)
  adding: results/exp1_colab/qrdata_w_info_Qwen2.5-7B-Instruct.json (deflated 85%)
  adding: results/exp1_colab/summary.json (deflated 54%)
Download /content/datamind_first3_results.zip from the Colab file browser.
